# Task 1: Data Collection and Preprocessing
## Scraping Google Play Store Reviews for Ethiopian Banks

### Banks Analyzed
- **Commercial Bank of Ethiopia (CBE)** — Ethiopia's largest state-owned bank
- **Bank of Abyssinia (BOA)** — Leading private bank
- **Dashen Bank** — Major private bank with strong mobile presence

### Methodology
Using `google-play-scraper` to collect user reviews, ratings, and dates.
Target: 400+ reviews per bank (1,200+ total).
Source: Google Play Store

In [7]:
import os
import pandas as pd
import numpy as np
from google_play_scraper import reviews, Sort
import warnings
warnings.filterwarnings('ignore')

os.chdir(r"C:\Users\pc\fintech-review-analytics")
print("All imports successful")

All imports successful


## Bank App IDs
Each app on Google Play has a unique package ID.
We use these to scrape reviews for each bank.

In [8]:
# Google Play Store App IDs for Ethiopian banks
banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensmart"
}

print("Target banks:")
for bank, app_id in banks.items():
    print(f"  {bank}: {app_id}")

Target banks:
  Commercial Bank of Ethiopia: com.combanketh.mobilebanking
  Bank of Abyssinia: com.boa.boaMobileBanking
  Dashen Bank: com.dashen.dashensmart


## Scraping Reviews
We collect 400+ reviews per bank using `google-play-scraper`.
Reviews are sorted by newest first to get the most recent feedback.
Each review includes: text, rating (1-5 stars), date, and username.

In [9]:
all_reviews = []

for bank_name, app_id in banks.items():
    print(f"\nScraping {bank_name}...")
    
    try:
        result, _ = reviews(
            app_id,
            lang='en',
            country='us',
            sort=Sort.NEWEST,
            count=600,  # request more to ensure 400+ after cleaning
            filter_score_with=None
        )
        
        for r in result:
            all_reviews.append({
                "review": r.get("content", ""),
                "rating": r.get("score", None),
                "date": r.get("at", None),
                "bank": bank_name,
                "source": "Google Play"
            })
        
        print(f"  Collected {len(result)} reviews")
        
    except Exception as e:
        print(f"  Error scraping {bank_name}: {e}")

print(f"\nTotal raw reviews collected: {len(all_reviews)}")


Scraping Commercial Bank of Ethiopia...
  Collected 600 reviews

Scraping Bank of Abyssinia...
  Collected 600 reviews

Scraping Dashen Bank...
  Collected 0 reviews

Total raw reviews collected: 1200


## Data Preprocessing
Steps:
1. Convert to DataFrame
2. Normalize dates to YYYY-MM-DD
3. Remove duplicates
4. Handle missing values
5. Export clean CSV

In [10]:
# Convert to DataFrame
df = pd.DataFrame(all_reviews)
print(f"Raw shape: {df.shape}")
print(f"\nMissing values before cleaning:")
print(df.isna().sum())

# Normalize dates
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Remove duplicates
before_dedup = len(df)
df = df.drop_duplicates(subset=["review", "bank"])
print(f"\nDuplicates removed: {before_dedup - len(df)}")

# Handle missing values
# Drop rows missing review text or rating
before_drop = len(df)
df = df.dropna(subset=["review", "rating"])
df = df[df["review"].str.strip() != ""]
print(f"Rows dropped (missing review/rating): {before_drop - len(df)}")

# Ensure correct column order
df = df[["review", "rating", "date", "bank", "source"]]

print(f"\nFinal shape: {df.shape}")
print(f"\nReviews per bank:")
print(df["bank"].value_counts())
print(f"\nRating distribution:")
print(df["rating"].value_counts().sort_index())

Raw shape: (1200, 5)

Missing values before cleaning:
review    0
rating    0
date      0
bank      0
source    0
dtype: int64

Duplicates removed: 246
Rows dropped (missing review/rating): 0

Final shape: (954, 5)

Reviews per bank:
bank
Bank of Abyssinia              499
Commercial Bank of Ethiopia    455
Name: count, dtype: int64

Rating distribution:
rating
1    262
2     35
3     58
4     74
5    525
Name: count, dtype: int64


## Data Quality Report

In [11]:
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

total = len(df)
print(f"\nTotal reviews: {total}")
print(f"Missing values: {df.isna().sum().sum()} ({df.isna().sum().sum()/total*100:.1f}%)")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

print(f"\nReviews per bank:")
for bank, count in df["bank"].value_counts().items():
    pct = count/total*100
    status = "✅" if count >= 400 else "⚠️"
    print(f"  {status} {bank}: {count} reviews ({pct:.1f}%)")

print(f"\nRating breakdown:")
for rating in sorted(df["rating"].unique()):
    count = len(df[df["rating"] == rating])
    print(f"  {int(rating)} stars: {count} reviews")

print(f"\nSample reviews:")
print(df.sample(3)[["bank", "rating", "review"]].to_string(index=False))

DATA QUALITY REPORT

Total reviews: 954
Missing values: 0 (0.0%)
Date range: 2024-11-12 to 2026-05-14

Reviews per bank:
  ✅ Bank of Abyssinia: 499 reviews (52.3%)
  ✅ Commercial Bank of Ethiopia: 455 reviews (47.7%)

Rating breakdown:
  1 stars: 262 reviews
  2 stars: 35 reviews
  3 stars: 58 reviews
  4 stars: 74 reviews
  5 stars: 525 reviews

Sample reviews:
                       bank  rating                              review
          Bank of Abyssinia       5                    it is a best app
Commercial Bank of Ethiopia       4 easy to use but hard to find easily
Commercial Bank of Ethiopia       1          notwork totransferto other


In [13]:
from google_play_scraper import search

results = search("Dashen Bank Ethiopia", lang="en", country="us", n_hits=5)
for r in results:
    print(f"Title: {r['title']}")
    print(f"App ID: {r['appId']}")
    print(f"Score: {r['score']}")
    print("---")

Title: Dashen Bank
App ID: com.dashen.dashensuperapp
Score: 4.14
---
Title: Dashen Bank Merchant
App ID: com.dashen.dashenmerchant
Score: None
---
Title: Dashen Mobile
App ID: com.cr2.amolelight
Score: 4.1267605
---
Title: Commercial Bank of Ethiopia
App ID: com.combanketh.mobilebanking
Score: 4.157025
---
Title: Dasheng Bank Mobile
App ID: com.cdb.ubx.chinadashengapplication
Score: None
---


In [14]:
# Scrape Dashen Bank with correct app ID
print("Scraping Dashen Bank with correct app ID...")

try:
    dashen_result, _ = reviews(
        "com.dashen.dashensuperapp",
        lang='en',
        country='us',
        sort=Sort.NEWEST,
        count=600,
        filter_score_with=None
    )
    
    dashen_reviews = []
    for r in dashen_result:
        dashen_reviews.append({
            "review": r.get("content", ""),
            "rating": r.get("score", None),
            "date": r.get("at", None),
            "bank": "Dashen Bank",
            "source": "Google Play"
        })
    
    print(f"Collected {len(dashen_result)} Dashen reviews")
    
    # Convert and clean
    dashen_df = pd.DataFrame(dashen_reviews)
    dashen_df["date"] = pd.to_datetime(dashen_df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    dashen_df = dashen_df.drop_duplicates(subset=["review", "bank"])
    dashen_df = dashen_df.dropna(subset=["review", "rating"])
    dashen_df = dashen_df[dashen_df["review"].str.strip() != ""]
    dashen_df = dashen_df[["review", "rating", "date", "bank", "source"]]
    
    print(f"Clean Dashen reviews: {len(dashen_df)}")
    
    # Merge with existing df
    df = pd.concat([df, dashen_df], ignore_index=True)
    print(f"\nTotal reviews now: {len(df)}")
    print(df["bank"].value_counts())

except Exception as e:
    print(f"Error: {e}")

Scraping Dashen Bank with correct app ID...
Collected 600 Dashen reviews
Clean Dashen reviews: 498

Total reviews now: 1452
bank
Bank of Abyssinia              499
Dashen Bank                    498
Commercial Bank of Ethiopia    455
Name: count, dtype: int64


In [15]:
df.to_csv("data/raw/bank_reviews.csv", index=False)
print(f"Updated CSV saved with {len(df)} total reviews")
print(df["bank"].value_counts())

Updated CSV saved with 1452 total reviews
bank
Bank of Abyssinia              499
Dashen Bank                    498
Commercial Bank of Ethiopia    455
Name: count, dtype: int64


In [12]:
# Save to data/raw/
os.makedirs("data/raw", exist_ok=True)
df.to_csv("data/raw/bank_reviews.csv", index=False)
print(f"Saved {len(df)} reviews to data/raw/bank_reviews.csv")
print("Remember: data/ is in .gitignore — never commit this file!")

Saved 954 reviews to data/raw/bank_reviews.csv
Remember: data/ is in .gitignore — never commit this file!
